In [16]:
!pip install --upgrade transformers==4.45.0 huggingface_hub

  Using cached huggingface_hub-1.6.0-py3-none-any.whl.metadata (13 kB)


In [17]:
import os
os.environ["WANDB_DISABLED"] = "true"
import re
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import warnings
warnings.filterwarnings("ignore")

We suggest using a single class, it will make refinement easier. 

In your implementation, feel free to update the training procedure, change model and do whatever feels right 

In [18]:
# =============================================================================
#  Stylometric feature extraction  (human vs AI code signals)
# =============================================================================

NUM_CODE_FEATURES = 7

_KEYWORDS = frozenset({
    'if', 'else', 'elif', 'for', 'while', 'return', 'def', 'class',
    'import', 'from', 'in', 'not', 'and', 'or', 'True', 'False', 'None',
    'try', 'except', 'finally', 'with', 'as', 'is', 'lambda', 'pass',
    'break', 'continue', 'raise', 'yield', 'del', 'global', 'nonlocal',
    'assert', 'async', 'await',
    'int', 'str', 'float', 'bool', 'list', 'dict', 'set', 'tuple',
    'print', 'range', 'len', 'self', 'super', 'type', 'isinstance',
    'var', 'let', 'const', 'function', 'public', 'private', 'protected',
    'static', 'void', 'new', 'this', 'null', 'undefined',
})


def extract_code_features(code: str) -> list:
    """
    Extract 7 hand-crafted stylometric features that help distinguish
    human-written code from AI-generated code.

    Features (all normalised to roughly [0, 1]):
      1. comment_ratio           – fraction of lines that are comments
      2. avg_line_length         – mean chars per non-empty line
      3. line_length_cv          – coefficient of variation of line lengths
      4. whitespace_consistency  – std-dev of leading whitespace
      5. avg_identifier_length   – mean length of user identifier tokens
      6. identifier_uniqueness   – unique identifiers / total identifiers
      7. max_nesting_depth       – deepest indentation level
    """
    lines = code.split('\n')
    non_empty = [l for l in lines if l.strip()]
    total_lines = max(len(lines), 1)
    ne_count = max(len(non_empty), 1)

    # 1 ── comment ratio  (AI tends to over-comment)
    comment_lines = sum(
        1 for l in lines
        if l.strip().startswith('#') or l.strip().startswith('//')
    )
    comment_ratio = comment_lines / total_lines

    # 2 ── average line length  (normalised, cap at 200)
    lengths = [len(l) for l in non_empty]
    mean_len = sum(lengths) / ne_count if lengths else 0.0
    avg_line_len = min(mean_len / 200.0, 1.0)

    # 3 ── line-length coefficient of variation  (AI is more uniform)
    if lengths and mean_len > 0:
        var = sum((x - mean_len) ** 2 for x in lengths) / ne_count
        cv = math.sqrt(var) / mean_len
    else:
        cv = 0.0
    line_len_cv = min(cv / 2.0, 1.0)

    # 4 ── whitespace consistency  (AI uses very regular indentation)
    leading = [len(l) - len(l.lstrip()) for l in non_empty]
    if leading:
        m = sum(leading) / len(leading)
        ws_var = sum((x - m) ** 2 for x in leading) / len(leading)
        ws_std = math.sqrt(ws_var)
    else:
        ws_std = 0.0
    ws_consistency = min(ws_std / 20.0, 1.0)

    # 5 & 6 ── identifier statistics  (AI favours longer, more descriptive names)
    identifiers = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]{0,60}\b', code)
    idents = [tok for tok in identifiers if tok not in _KEYWORDS]
    if idents:
        avg_id_len = sum(len(i) for i in idents) / len(idents)
        unique_ratio = len(set(idents)) / len(idents)
    else:
        avg_id_len = 0.0
        unique_ratio = 0.0
    avg_ident_len = min(avg_id_len / 20.0, 1.0)

    # 7 ── max nesting depth  (normalised, 32 spaces ≈ 8 indent levels)
    max_indent = max(leading) if leading else 0
    max_depth = min(max_indent / 32.0, 1.0)

    return [
        comment_ratio,
        avg_line_len,
        line_len_cv,
        ws_consistency,
        avg_ident_len,
        unique_ratio,
        max_depth,
    ]


class CodeFeatureCollator(DataCollatorWithPadding):
    """Extends the default padding collator to also batch ``code_features``."""

    def __call__(self, features):
        code_feats = None
        if features and 'code_features' in features[0]:
            code_feats = [f.pop('code_features') for f in features]
        batch = super().__call__(features)
        if code_feats is not None:
            batch['code_features'] = torch.tensor(code_feats, dtype=torch.float32)
        return batch


# =============================================================================
#  BiLSTM + Dense-block classification head  (backbone-agnostic)
# =============================================================================

class BiLSTMClassificationModel(nn.Module):
    """
    Any RoBERTa-family backbone -> BiLSTM -> Dense blocks -> classifier.
    Concatenates hand-crafted stylometric features with the pooled LSTM
    output before the dense blocks.
    Works with CodeBERT, UniXcoder, etc.
    """

    def __init__(
        self,
        model_name: str,
        num_labels: int = 2,
        lstm_hidden_size: int = 256,
        lstm_num_layers: int = 2,
        dropout: float = 0.3,
        num_code_features: int = NUM_CODE_FEATURES,
    ):
        super().__init__()
        self.num_labels = num_labels
        self.num_code_features = num_code_features

        # ---------- Transformer backbone ----------
        self.transformer = RobertaModel.from_pretrained(model_name)
        self.config = self.transformer.config
        hidden_size = self.config.hidden_size                  # 768

        # ---------- BiLSTM head ----------
        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden_size,
            num_layers=lstm_num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_num_layers > 1 else 0.0,
        )
        lstm_out_size = lstm_hidden_size * 2                   # bidirectional

        # ---------- Dense blocks (FC + BN + GELU + Dropout) ----------
        dense_input = lstm_out_size + num_code_features   # concat stylometric feats
        self.dense_block = nn.Sequential(
            nn.Linear(dense_input, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ---------- Classifier ----------
        self.classifier = nn.Linear(128, num_labels)

    def forward(self, input_ids=None, attention_mask=None, labels=None,
                code_features=None, **kwargs):
        transformer_out = self.transformer(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        seq_out = transformer_out.last_hidden_state              # (B, L, H)

        lstm_out, _ = self.bilstm(seq_out)                       # (B, L, 2*lstm_h)

        # Masked mean-pooling
        if attention_mask is not None:
            mask = attention_mask.unsqueeze(-1).float()
            pooled = (lstm_out * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)
        else:
            pooled = lstm_out.mean(dim=1)

        # Concatenate hand-crafted stylometric features
        if code_features is not None:
            pooled = torch.cat([pooled, code_features.float()], dim=-1)

        logits = self.classifier(self.dense_block(pooled))

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


# =============================================================================
#  Generic Trainer  (plain  OR  BiLSTM variant, any backbone)
# =============================================================================

class CodeClassifierTrainer:
    """
    Unified trainer that handles:
      - Any RoBERTa-family backbone  (CodeBERT / UniXcoder)
      - Plain head  (RobertaForSequenceClassification)
      - BiLSTM head (BiLSTMClassificationModel + stylometric features)
    """

    def __init__(
        self,
        model_name: str = "microsoft/codebert-base",
        use_bilstm: bool = False,
        max_length: int = 512,
        lstm_hidden_size: int = 256,
        lstm_num_layers: int = 2,
        dropout: float = 0.3,
    ):
        self.model_name = model_name
        self.use_bilstm = use_bilstm
        self.max_length = max_length
        self.lstm_hidden_size = lstm_hidden_size
        self.lstm_num_layers = lstm_num_layers
        self.dropout = dropout
        self.tokenizer = None
        self.model = None
        self.num_labels = None

    @property
    def tag(self):
        base = self.model_name.split("/")[-1]
        return f"{base}{'+BiLSTM' if self.use_bilstm else ''}"

    def load_and_prepare_data(self, sample_size=None, val_sample_size=None, random_seed=42):
        try:
            df = pd.read_parquet(
                '/kaggle/input/datasets/daniilor/semeval-2026-task13/'
                'SemEval-2026-Task13/task_a/task_a_training_set_1.parquet'
            )
            print(f"[{self.tag}] Dataset columns: {df.columns.tolist()}")
            print(f"[{self.tag}] Original dataset size: {len(df)}")

            if 'code' not in df.columns or 'label' not in df.columns:
                raise ValueError("Dataset must contain 'code' and 'label' columns")

            df = df.dropna(subset=['code', 'label'])
            df['label'] = df['label'].astype(int)

            if sample_size is not None and sample_size < len(df):
                df = df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(sample_size * len(x) / len(df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled train size: {len(df)} (seed={random_seed})")

            self.num_labels = df['label'].nunique()
            print(f"[{self.tag}] Number of unique labels: {self.num_labels}")
            print(f"[{self.tag}] Train label distribution:\n{df['label'].value_counts().sort_index()}")

            val_df = pd.read_parquet(
                '/kaggle/input/datasets/daniilor/semeval-2026-task13/'
                'SemEval-2026-Task13/task_a/task_a_validation_set.parquet'
            )
            val_df = val_df.dropna(subset=['code', 'label'])
            val_df['label'] = val_df['label'].astype(int)

            if val_sample_size is not None and val_sample_size < len(val_df):
                val_df = val_df.groupby('label', group_keys=False).apply(
                    lambda x: x.sample(
                        n=max(1, int(val_sample_size * len(x) / len(val_df))),
                        random_state=random_seed,
                    )
                ).reset_index(drop=True)
                print(f"[{self.tag}] Sampled val size: {len(val_df)} (seed={random_seed})")

            print(f"[{self.tag}] Val label distribution:\n{val_df['label'].value_counts().sort_index()}")
            print(f"[{self.tag}] Train samples: {len(df)}, Validation samples: {len(val_df)}")
            return df, val_df

        except Exception as e:
            print(f"[{self.tag}] Error loading dataset: {e}")
            raise

    def initialize_model_and_tokenizer(self):
        print(f"[{self.tag}] Initializing model and tokenizer ...")
        self.tokenizer = RobertaTokenizer.from_pretrained(self.model_name)

        if self.use_bilstm:
            self.model = BiLSTMClassificationModel(
                model_name=self.model_name,
                num_labels=self.num_labels,
                lstm_hidden_size=self.lstm_hidden_size,
                lstm_num_layers=self.lstm_num_layers,
                dropout=self.dropout,
                num_code_features=NUM_CODE_FEATURES,
            ).to('cuda')
        else:
            self.model = RobertaForSequenceClassification.from_pretrained(
                self.model_name,
                num_labels=self.num_labels,
                problem_type="single_label_classification",
            ).to('cuda')

        total = sum(p.numel() for p in self.model.parameters())
        train = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"[{self.tag}] {self.num_labels} labels | params {total:,} | trainable {train:,}")

    def tokenize_function(self, examples):
        return self.tokenizer(
            examples['code'],
            truncation=True,
            padding=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

    def prepare_datasets(self, train_df, val_df):
        print(f"[{self.tag}] Preparing datasets ...")
        train_dataset = Dataset.from_pandas(train_df[['code', 'label']])
        val_dataset   = Dataset.from_pandas(val_df[['code', 'label']])

        # Compute stylometric features for BiLSTM variants
        if self.use_bilstm:
            print(f"[{self.tag}] Extracting {NUM_CODE_FEATURES} stylometric features ...")
            train_dataset = train_dataset.map(
                lambda ex: {'code_features': extract_code_features(ex['code'])},
            )
            val_dataset = val_dataset.map(
                lambda ex: {'code_features': extract_code_features(ex['code'])},
            )

        train_dataset = train_dataset.map(self.tokenize_function, batched=True, remove_columns=['code'])
        val_dataset   = val_dataset.map(self.tokenize_function,   batched=True, remove_columns=['code'])

        train_dataset = train_dataset.rename_column('label', 'labels')
        val_dataset   = val_dataset.rename_column('label', 'labels')
        return train_dataset, val_dataset

    def compute_metrics(self, eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        accuracy = accuracy_score(labels, predictions)
        precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
        return {'accuracy': accuracy, 'f1': f1, 'precision': precision, 'recall': recall}

    def train(self, train_dataset, val_dataset,
              output_dir="./results", num_epochs=3,
              batch_size=16, learning_rate=2e-5):
        print(f"[{self.tag}] Starting training ...")

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=num_epochs,
            per_device_train_batch_size=batch_size,
            per_device_eval_batch_size=batch_size,
            weight_decay=0.01,
            logging_dir=f'{output_dir}/logs',
            logging_steps=5,
            eval_strategy="steps",
            eval_steps=500,
            save_strategy="steps",
            save_steps=500,
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            greater_is_better=True,
            remove_unused_columns=False,
            learning_rate=learning_rate,
            lr_scheduler_type="linear",
            save_total_limit=2,
            report_to=[],
        )

        if self.use_bilstm:
            data_collator = CodeFeatureCollator(tokenizer=self.tokenizer)
        else:
            data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        trainer = Trainer(
            model=self.model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            tokenizer=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=self.compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
        )
        print(f"[{self.tag}] Start training")
        trainer.train()

        trainer.save_model()
        self.tokenizer.save_pretrained(output_dir)
        print(f"[{self.tag}] Training completed. Model saved to {output_dir}")
        return trainer

    def evaluate_model(self, trainer, val_dataset):
        print(f"[{self.tag}] Evaluating ...")
        predictions = trainer.predict(val_dataset)
        y_pred = np.argmax(predictions.predictions, axis=1)
        y_true = predictions.label_ids
        print(f"[{self.tag}] Classification Report:")
        print(classification_report(y_true, y_pred))
        return predictions

    def run_full_pipeline(self, output_dir=None,
                          num_epochs=3, batch_size=16, learning_rate=2e-5,
                          sample_size=2000, val_sample_size=500, random_seed=42):
        if output_dir is None:
            output_dir = f"./results_{self.tag.replace('+', '_')}"
        try:
            train_df, val_df = self.load_and_prepare_data(
                sample_size=sample_size,
                val_sample_size=val_sample_size,
                random_seed=random_seed,
            )
            self.initialize_model_and_tokenizer()
            train_dataset, val_dataset = self.prepare_datasets(train_df, val_df)
            trainer = self.train(
                train_dataset, val_dataset,
                output_dir=output_dir,
                num_epochs=num_epochs,
                batch_size=batch_size,
                learning_rate=learning_rate,
            )
            self.evaluate_model(trainer, val_dataset)
            print(f"[{self.tag}] Pipeline completed successfully!")
            return trainer
        except Exception as e:
            print(f"[{self.tag}] Error in pipeline: {e}")
            raise

print("extract_code_features, CodeFeatureCollator, BiLSTMClassificationModel, CodeClassifierTrainer defined.")

extract_code_features, CodeFeatureCollator, BiLSTMClassificationModel, CodeClassifierTrainer defined.


In [19]:
from tqdm import tqdm

# =============================================================================
#  Test-set prediction  (returns DataFrame with predictions attached)
# =============================================================================

@torch.no_grad()
def predict_on_test(trainer_obj, parquet_path,
                    max_length=256, batch_size=32, device=None):
    """
    Run inference on a test parquet.
    Returns the original DataFrame with an added ``prediction`` column.
    """
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = trainer_obj.tokenizer
    model = trainer_obj.model
    model.to(device)
    model.eval()

    df = pd.read_parquet(parquet_path)
    if 'code' not in df.columns:
        raise ValueError(f"Parquet must have a 'code' column. Found: {df.columns.tolist()}")
    df = df.dropna(subset=['code']).reset_index(drop=True)
    codes = df['code'].tolist()

    all_preds = []
    is_bilstm = isinstance(model, BiLSTMClassificationModel)

    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch_codes = codes[i:i + batch_size]
        enc = tokenizer(
            batch_codes,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )
        fwd_kwargs = dict(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        )
        if is_bilstm:
            feats = [extract_code_features(c) for c in batch_codes]
            fwd_kwargs['code_features'] = torch.tensor(
                feats, dtype=torch.float32,
            ).to(device)
        logits = model(**fwd_kwargs).logits
        all_preds.extend(logits.argmax(dim=-1).cpu().tolist())

    df['prediction'] = all_preds
    return df


# =============================================================================
#  Category-wise evaluation on test set
# =============================================================================

# Language grouping
SEEN_LANGS   = {'c++', 'cpp', 'python', 'java'}
UNSEEN_LANGS = {'go', 'php', 'c#', 'csharp', 'c', 'javascript', 'js'}

# Domain grouping
SEEN_DOMAINS   = {'algorithmic'}
UNSEEN_DOMAINS = {'research', 'production'}


def _normalise(val):
    """Lower-case + strip for robust matching."""
    return str(val).strip().lower()


def evaluate_by_category(df, tag="Model"):
    """
    Given a DataFrame with columns ``label``, ``prediction``, ``language``,
    and ``domain`` (or ``task_type``), print classification metrics for the
    four evaluation settings:

      (i)   Seen Languages   & Seen Domains
      (ii)  Unseen Languages & Seen Domains
      (iii) Seen Languages   & Unseen Domains
      (iv)  Unseen Languages & Unseen Domains
    """
    # --- robust column detection ---
    lang_col = None
    for c in df.columns:
        if c.lower() in ('language', 'lang', 'programming_language'):
            lang_col = c
            break
    domain_col = None
    for c in df.columns:
        if c.lower() in ('domain', 'task_type', 'category'):
            domain_col = c
            break

    if lang_col is None or domain_col is None:
        print(f"[{tag}] WARNING: Could not find language (found={lang_col}) or "
              f"domain (found={domain_col}) column.  Columns: {df.columns.tolist()}")
        print(f"[{tag}] Falling back to overall evaluation.")
        print(classification_report(df['label'], df['prediction']))
        return

    df['_lang']   = df[lang_col].apply(_normalise)
    df['_domain'] = df[domain_col].apply(_normalise)

    settings = [
        ("(i)   Seen Lang & Seen Domain",   SEEN_LANGS,   SEEN_DOMAINS),
        ("(ii)  Unseen Lang & Seen Domain",  UNSEEN_LANGS, SEEN_DOMAINS),
        ("(iii) Seen Lang & Unseen Domain",  SEEN_LANGS,   UNSEEN_DOMAINS),
        ("(iv)  Unseen Lang & Unseen Domain", UNSEEN_LANGS, UNSEEN_DOMAINS),
    ]

    print(f"\n{'=' * 70}")
    print(f"  TEST EVALUATION  --  {tag}")
    print(f"{'=' * 70}")

    summary = {}
    for name, langs, domains in settings:
        mask = df['_lang'].isin(langs) & df['_domain'].isin(domains)
        subset = df[mask]
        n = len(subset)
        if n == 0:
            print(f"\n  {name}:  ** no samples **")
            summary[name] = {'n': 0, 'accuracy': None, 'f1': None}
            continue

        y_true = subset['label'].values
        y_pred = subset['prediction'].values
        acc = accuracy_score(y_true, y_pred)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0,
        )
        print(f"\n  {name}  (n={n})")
        print(f"    Accuracy : {acc:.4f}")
        print(f"    Precision: {prec:.4f}")
        print(f"    Recall   : {rec:.4f}")
        print(f"    F1       : {f1:.4f}")
        print(classification_report(y_true, y_pred, zero_division=0))
        summary[name] = {'n': n, 'accuracy': acc, 'f1': f1}

    # overall
    acc_all = accuracy_score(df['label'], df['prediction'])
    _, _, f1_all, _ = precision_recall_fscore_support(
        df['label'], df['prediction'], average='weighted', zero_division=0,
    )
    print(f"\n  OVERALL  (n={len(df)})")
    print(f"    Accuracy : {acc_all:.4f}    F1 : {f1_all:.4f}")
    print("=" * 70)
    summary['overall'] = {'n': len(df), 'accuracy': acc_all, 'f1': f1_all}

    df.drop(columns=['_lang', '_domain'], inplace=True)
    return summary

print("predict_on_test(), evaluate_by_category() defined.")

predict_on_test(), evaluate_by_category() defined.


In [ ]:
# =============================================================================
#  Run CodeBERT on full dataset  (call from notebook)
# =============================================================================

def run_codebert_full(
    num_epochs=10,
    batch_size=16,
    learning_rate=2e-5,
    max_length=256,
    random_seed=42,
    test_parquet="/kaggle/input/datasets/daniilor/semeval-2026-task13/"
                 "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet",
    output_base="/kaggle/working",
):
    """
    Train on FULL dataset and evaluate:
      1. CodeBERT  (plain / barebones)
    Then report test-set results for all 4 evaluation categories.
    """
    trainer_obj = CodeClassifierTrainer(
        model_name="microsoft/codebert-base",
        use_bilstm=False,
        max_length=max_length,
        dropout=0.3,
    )
    tag = trainer_obj.tag
    out_dir = os.path.join(output_base, f"results_{tag.replace('+', '_')}")

    print("\n" + "=" * 70)
    print(f"  MODEL: {tag}")
    print("=" * 70)

    # --- train on FULL dataset (sample_size=None) ---
    hf_trainer = trainer_obj.run_full_pipeline(
        output_dir=out_dir,
        num_epochs=num_epochs,
        batch_size=batch_size,
        learning_rate=learning_rate,
        sample_size=None,           # full training set
        val_sample_size=None,       # full validation set
        random_seed=random_seed,
    )

    # --- test-set predictions + category evaluation ---
    test_df = predict_on_test(
        trainer_obj=trainer_obj,
        parquet_path=test_parquet,
        max_length=max_length,
        batch_size=batch_size * 2,
        device="cuda",
    )

    # save CSV
    csv_path = os.path.join(output_base, f"submission_{tag.replace('+', '_')}.csv")
    id_col = 'ID' if 'ID' in test_df.columns else 'index'
    test_df[[id_col, 'prediction'] if id_col in test_df.columns
            else ['prediction']].to_csv(csv_path, index=(id_col == 'index'))
    print(f"Predictions saved to {csv_path}  ({len(test_df)} rows)")

    # category-wise evaluation (only if test set has labels)
    if 'label' in test_df.columns:
        cat_summary = evaluate_by_category(test_df, tag=tag)
    else:
        print(f"[{tag}] Test set has no 'label' column — skipping evaluation.")
        cat_summary = None

    results_summary = {
        tag: {
            "best_val_f1": (hf_trainer.state.best_metric
                            if hf_trainer.state.best_metric else "N/A"),
            "csv": csv_path,
            "categories": cat_summary,
        }
    }

    # --- final summary ---
    print("\n\n" + "=" * 70)
    print("  RESULTS  --  CodeBERT (full dataset)")
    print("=" * 70)
    info = results_summary[tag]
    print(f"\n  {tag}")
    print(f"    Best val F1 : {info['best_val_f1']}")
    if info['categories']:
        for cat, m in info['categories'].items():
            if m.get('f1') is not None:
                print(f"    {cat:45s}  acc={m['accuracy']:.4f}  f1={m['f1']:.4f}  n={m['n']}")
    print("=" * 70)

    return results_summary

print("run_codebert_full() defined.")

run_two_models() defined.


In [ ]:
# =============================================================================
#  Run CodeBERT on full dataset  --  just hit Run!
# =============================================================================

results = run_codebert_full(
    num_epochs=10,
    batch_size=16,
    learning_rate=2e-5,
    max_length=256,
    random_seed=42,
)


  MODEL: codebert-base
[codebert-base] Dataset columns: ['code', 'generator', 'label', 'language']
[codebert-base] Original dataset size: 500000
[codebert-base] Number of unique labels: 2
[codebert-base] Train label distribution:
label
0    238475
1    261525
Name: count, dtype: int64
[codebert-base] Val label distribution:
label
0    47695
1    52305
Name: count, dtype: int64
[codebert-base] Train samples: 500000, Validation samples: 100000
[codebert-base] Initializing model and tokenizer ...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/codebert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[codebert-base] 2 labels | params 124,647,170 | trainable 124,647,170
[codebert-base] Preparing datasets ...


Map:   0%|          | 0/500000 [00:00<?, ? examples/s]